# Exploração dos Parquets — Coorte, Feature Engineering e Trajetórias até T0

## Qual tabela representa a coorte com feature engineering até T0?

**`outputs/target_trial/processed/trial_dataset.parquet`** — essa é a tabela principal.

Ela é gerada por `scripts/target_trial/build_target_trial.py` e contém:
- **1.484 pacientes elegíveis** (506 tratados + 978 controles)
- **T0** ancorado no momento da primeira transfusão (tratados) ou pseudo-T0 baseado na mediana do offset (controles)
- **Features de janela pré-T0** (48 h): hemoglobin, lactate, creatinine, platelets, heart\_rate, mbp, resp\_rate, spo2, sofa, vasopressores, ventilação — cada uma com `_mean`, `_median`, `_min`, `_max`, `_std`, `_first`, `_last`, `_delta`, `_slope`
- **Desfechos**: mortality\_anytime, ventilation\_hours, rrt\_on, any\_vasopressor, icu\_los\_hours

---

### Mapa dos parquets neste notebook

| # | Arquivo | Papel no pipeline |
|---|---------|-------------------|
| 1 | `processed/eligibility.parquet` | Critérios de elegibilidade por paciente |
| 2 | `processed/treatment_assignment.parquet` | Atribuição de tratamento + T0 |
| **3** | **`processed/trial_dataset.parquet`** | **Coorte final: features pré-T0 + desfechos** |
| 4 | `causal/nuisance_predictions.parquet` | Propensity score + predições de outcome |
| 5 | `final_groups/final_group_assignments.parquet` | Grupos de heterogeneidade (CATE) |
| 6 | `final_groups/final_group_patient_counterfactuals.parquet` | Contrafactuais individuais |
| 7 | `legacy_crosswalk/legacy_k2_assignments.parquet` | Crosswalk K=2 legado |
| 8 | `legacy_rule_crosswalk/run_cal03_noreplace_w48/legacy_rule_assignments.parquet` | Grupos legado (sem reposição) |
| 9 | `legacy_rule_crosswalk/run_cal03_replace_full_w48/legacy_rule_assignments.parquet` | Grupos legado (com reposição) |
| 10 | `vcip_lite/vcip_lite_individual_counterfactuals.parquet` | VCIP-lite: Y(0), Y(1), ITE por paciente |

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 30)

BASE = Path('outputs/target_trial')

PARQUETS = {
    'eligibility':                    BASE / 'processed/eligibility.parquet',
    'treatment_assignment':           BASE / 'processed/treatment_assignment.parquet',
    'trial_dataset':                  BASE / 'processed/trial_dataset.parquet',
    'nuisance_predictions':           BASE / 'causal/nuisance_predictions.parquet',
    'final_group_assignments':        BASE / 'final_groups/final_group_assignments.parquet',
    'final_group_patient_counterfactuals': BASE / 'final_groups/final_group_patient_counterfactuals.parquet',
    'legacy_k2_assignments':          BASE / 'legacy_crosswalk/legacy_k2_assignments.parquet',
    'legacy_rule_noreplace':          BASE / 'legacy_rule_crosswalk/run_cal03_noreplace_w48/legacy_rule_assignments.parquet',
    'legacy_rule_replace':            BASE / 'legacy_rule_crosswalk/run_cal03_replace_full_w48/legacy_rule_assignments.parquet',
    'vcip_lite_counterfactuals':      BASE / 'vcip_lite/vcip_lite_individual_counterfactuals.parquet',
}

def show(name: str, path: Path, n: int = 5) -> None:
    sep = '─' * 80
    print(f'\n{sep}')
    print(f'  {name}')
    print(f'  {path}')
    print(sep)
    if not path.exists():
        print('  ⚠ arquivo não encontrado')
        return
    df = pd.read_parquet(path)
    print(f'  shape: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
    print(f'  colunas: {list(df.columns)}')
    dtypes = df.dtypes.value_counts().to_dict()
    print(f'  dtypes: {dtypes}')
    print()
    display(df.head(n))

print('Ambiente pronto. Parquets configurados:', len(PARQUETS))

---
## 1 — `eligibility.parquet`
Tabela intermediária: um registro por paciente (`stay_id`) com flags booleanos de cada critério de elegibilidade.
Usada para auditar quem foi incluído/excluído e o motivo.

In [ ]:
show('eligibility', PARQUETS['eligibility'])

---
## 2 — `treatment_assignment.parquet`
Um registro por paciente elegível com:
- `transfused` (0/1) — se recebeu transfusão
- `t0` / `t0_transf` — timestamp do T0
- `pseudo_t0` — flag indicando se o T0 é pseudo (controles) ou real (tratados)

In [ ]:
show('treatment_assignment', PARQUETS['treatment_assignment'])

---
## 3 ⭐ — `trial_dataset.parquet` — TABELA PRINCIPAL
**Esta é a coorte final com feature engineering e trajetórias até T0.**

Cada linha é um paciente elegível. As colunas incluem:
- Identificação (`stay_id`), tratamento (`transfused`), T0
- Features estáticas: `age`, `sex`, `bmi`
- Features temporais pré-T0 (janela 48 h), para cada variável fisiológica:
  `_mean`, `_median`, `_min`, `_max`, `_std`, `_first`, `_last`, `_delta`, `_slope`
- Desfechos: `mortality_anytime`, `ventilation_hours`, `rrt_on`, etc.
- Split de validação: `split`

In [ ]:
show('trial_dataset ⭐', PARQUETS['trial_dataset'])

In [ ]:
# Inspecionar grupos de colunas do trial_dataset
df_trial = pd.read_parquet(PARQUETS['trial_dataset'])

feature_cols = [c for c in df_trial.columns if any(
    c.endswith(s) for s in ['_mean','_median','_min','_max','_std','_first','_last','_delta','_slope']
)]
outcome_cols = [c for c in df_trial.columns if c in [
    'mortality_anytime','ventilation_hours','rrt_on','any_vasopressor','nee_mcgkgmin_max','icu_los_hours'
]]
id_cols = [c for c in df_trial.columns if c not in feature_cols + outcome_cols]

print(f'Colunas de identificação/tratamento/T0 ({len(id_cols)}): {id_cols}')
print(f'\nFeatures pré-T0 ({len(feature_cols)}): {feature_cols}')
print(f'\nDesfechos ({len(outcome_cols)}): {outcome_cols}')
print(f'\nDistribuição tratamento:')
print(df_trial['transfused'].value_counts().rename({0: 'controle', 1: 'tratado'}))
if 'pseudo_t0' in df_trial.columns:
    print(f'\nPseudo-T0 (controles): {df_trial["pseudo_t0"].sum()}')

---
## 4 — `nuisance_predictions.parquet`
Predições dos modelos de nuisance para inferência causal:
- `propensity_score` (probabilidade de receber transfusão)
- `outcome_untreated` / `outcome_treated` (Y(0) e Y(1) preditos)
- Usados como inputs para estimadores AIPW/DR

In [ ]:
show('nuisance_predictions', PARQUETS['nuisance_predictions'])

---
## 5 — `final_group_assignments.parquet`
Grupos de heterogeneidade de efeito de tratamento (CATE).
Cada paciente recebe um grupo (benefício / neutro / risco) com base no ITE estimado.

In [ ]:
show('final_group_assignments', PARQUETS['final_group_assignments'])

---
## 6 — `final_group_patient_counterfactuals.parquet`
Contrafactuais individuais por paciente: Y(0), Y(1) e ITE estimados no modelo final.

In [ ]:
show('final_group_patient_counterfactuals', PARQUETS['final_group_patient_counterfactuals'])

---
## 7 — `legacy_k2_assignments.parquet`
Crosswalk com os grupos K=2 do pipeline legado (artigo anterior), para comparação de consistência.

In [ ]:
show('legacy_k2_assignments', PARQUETS['legacy_k2_assignments'])

---
## 8 — `legacy_rule_assignments` — run sem reposição (cal03_noreplace_w48)

In [ ]:
show('legacy_rule_noreplace', PARQUETS['legacy_rule_noreplace'])

---
## 9 — `legacy_rule_assignments` — run com reposição (cal03_replace_full_w48)

In [ ]:
show('legacy_rule_replace', PARQUETS['legacy_rule_replace'])

---
## 10 — `vcip_lite_individual_counterfactuals.parquet`
Versão leve do VCIP: Y(0), Y(1) e ITE por paciente usando modelo simplificado (sem embeddings temporais).

In [ ]:
show('vcip_lite_counterfactuals', PARQUETS['vcip_lite_counterfactuals'])

---
## Resumo de shapes
Visão geral de quantas linhas/colunas cada parquet tem.

In [ ]:
summary = []
for name, path in PARQUETS.items():
    if path.exists():
        df = pd.read_parquet(path)
        summary.append({'parquet': name, 'linhas': df.shape[0], 'colunas': df.shape[1],
                        'tamanho_kb': round(path.stat().st_size / 1024, 1)})
    else:
        summary.append({'parquet': name, 'linhas': None, 'colunas': None, 'tamanho_kb': None})

display(pd.DataFrame(summary).set_index('parquet'))